# Cold-start transfer learning: how much patient history is needed?

Test whether population transfer offers a larger advantage when little target-patient history is available. This is a **new, isolated follow-up experiment**, not a replacement for the completed full-history benchmark and not a guarantee of larger gains.

**Default:** GRU, 30-minute horizon, 12 patients, seeds 41/42/43, and **1 / 3 / 7 days / full history**. The official held-out test set stays fixed. Includes MAE, RMSE, persistence, paired patient/seed uncertainty, training histories and predictions.

**Use a GPU runtime** (your A100 is suitable). This notebook contains the runner and snapshots of the repository's XML loader/preprocessor, so no repository clone or pushed changes are required. It reads your existing OhioT1DM XML files from Drive. Previously trained full-history checkpoints cannot be reused: they may have seen target data outside the limited-history budget.

Completed pretraining and each paired patient/seed/budget job are mirrored to Drive and reused after a disconnect when the data, code, configuration and package-version fingerprint matches.


## 1. Drive and experiment configuration

The budget counts **elapsed days, including validation**, ending at the last available training CGM reading. Using the most recent history controls recency across budgets. This simulates a newly available, short patient record; it is not a prospective newly diagnosed-patient cohort.

The primary follow-up is **MAE with 1 day versus this experiment's full-history control** at 30 minutes. The other budgets and RMSE are reported, even if they show no gain. Keep these settings fixed after inspecting test results.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_DATA = Path('/content/drive/MyDrive/ohiot1dm')  # contains 2018/ and 2020/
DRIVE_RESULTS = Path('/content/drive/MyDrive/bg-results')
LOCAL_ROOT = Path('/content/cold_start_followup')
SMOKE_TEST = False  # True: two patients, one seed, tiny model, one epoch; NOT paper results
MODEL = 'gru'
HORIZON_MINUTES = 30
BUDGETS_DAYS = [1, 3, 7, 'full']
SEEDS = [41, 42, 43]


## 2. What is controlled

- **Four inputs:** glucose, basal insulin, bolus, carbohydrates; 60-minute input window. Predict all horizon steps, score only the final step.
- **Source pretraining:** all other patients' official training records; target patient excluded. Each source is split chronologically 80/20. Source-only validation selects the checkpoint. The same checkpoint is reused across every target budget for that patient/seed.
- **Patient personalization:** RL starts from random weights; TL starts from the source checkpoint. Both use the exact same target data, normalization, chronological validation, batch ordering seed, learning rate, 50-epoch target cap and patience 10. Both reset Adam at the start of target training. All TL layers are trainable.
- **No temporal leakage:** slice raw histories before preprocessing; fit normalization on the fitting portion only; construct train and validation windows separately. Windows must have exact five-minute spacing and valid features. No interpolation across gaps. Each source has its own fitting-only normalization; RL and TL share the target's fitting-only normalization.
- **Fixed evaluation:** identical official test windows across budgets, modes and seeds; test data never select epochs or hyperparameters. Persistence is checked for invariance. Source records are treated as an already available historical library; their calendar dates are not restricted to the target date.
- **A separate full-history control:** rerun with this same protocol. Chronological validation, source-only checkpoint selection, gap checks and matched target-training caps differ from the published experiment. Comparisons to the article's numbers are therefore descriptive, not a clean isolation of history length. Compare budgets within this notebook.

Strict evaluation does not by itself prove why a result is small, and a cold-start gain cannot explain the change between two prior estimates. The manuscript's revised headline also reflects corrected calculations and changed saved configurations, not just added seeds. This notebook leaves the article unchanged.


In [ ]:
import os, sys, subprocess, json, shutil, hashlib, importlib.util
os.environ.setdefault('MPLCONFIGDIR', '/content/cold-start-matplotlib')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
               'numpy>=1.24,<3', 'pandas>=2,<3', 'scipy>=1.10,<2', 'matplotlib>=3.7,<4'], check=True)
import torch
if not torch.cuda.is_available() and not SMOKE_TEST:
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then reconnect.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU (smoke only)')


## 3. Code from the repository

The runner (`RUN/experiments/run_cold_start.py`) and the two preprocessing
modules (`benchmark/data/loaders.py`, `benchmark/data/preprocessors.py`) are
loaded from a checkout rather than embedded here, so this notebook cannot drift
from the code the repository documents. `protocol.json` still records a SHA-256
for each of them with every run, and because the embedded copies were exact
snapshots, the run fingerprint is unchanged — runs started under the previous
version of this notebook still resume.

The branch must carry the runner and, for section 7, the plotting script. If
they are not pushed yet, set `SOURCE = 'drive'` and put a copy of the repository
folder in Drive instead.

This notebook keeps its own output tree (`cold_start_followup/`) and does not
use `run_on_colab.ipynb`'s experiment folders.


In [ ]:
REPO_DIR   = Path('/content/BG-forecasting')
SOURCE     = 'git'          # 'drive' copies DRIVE_REPO instead of cloning
REPO_URL   = 'https://github.com/beatriz-fulgencio/BG-forecasting.git'
BRANCH     = 'bench2'
DRIVE_REPO = Path('/content/drive/MyDrive/BG-forecasting')

if SOURCE == 'git':
    if not REPO_DIR.is_dir():
        subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        pull = subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print('WARNING: git pull --ff-only failed, so this checkout may be stale.\n'
                  + (pull.stderr or pull.stdout).strip() + '\n')
elif SOURCE == 'drive':
    if not REPO_DIR.is_dir():
        shutil.copytree(DRIVE_REPO, REPO_DIR)
else:
    raise ValueError("SOURCE must be 'git' or 'drive'")

RUNNER      = REPO_DIR / 'RUN' / 'experiments' / 'run_cold_start.py'
SUPPORT_DIR = REPO_DIR / 'benchmark' / 'data'        # loaders.py + preprocessors.py
PLOT_SCRIPT = REPO_DIR / 'RUN' / 'experiments' / 'plot_cold_start_tl_vs_rl.py'

# Fail here, naming the file, rather than after the data is staged. The plotting
# script is only needed by section 7, so its absence is a note, not an error.
missing = [str(p) for p in (RUNNER, SUPPORT_DIR / 'loaders.py', SUPPORT_DIR / 'preprocessors.py')
           if not p.is_file()]
if missing:
    raise SystemExit('Missing from this checkout:\n  ' + '\n  '.join(missing) +
                     '\n\nPush them to ' + BRANCH + ", or set SOURCE = 'drive'.")
if not PLOT_SCRIPT.is_file():
    print(f'NOTE: {PLOT_SCRIPT.name} is not in this checkout; sections 1-6 run, section 7 will not.')

# The support modules import nothing from the benchmark package, so they load by
# path exactly as the embedded snapshots did -- no editable install needed.
spec = importlib.util.spec_from_file_location('cold_start', RUNNER)
cold = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cold)

head = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'],
                      capture_output=True, text=True).stdout.strip()
print('runner :', RUNNER)
print('support:', SUPPORT_DIR)
print('commit :', head or '(not a git checkout)')

import copy
CFG = copy.deepcopy(cold.DEFAULTS)
assert HORIZON_MINUTES > 0 and HORIZON_MINUTES % 5 == 0
CFG.update(model=MODEL, horizon_steps=HORIZON_MINUTES//5,
           budgets_days=BUDGETS_DAYS, seeds=SEEDS, smoke=SMOKE_TEST)
if SMOKE_TEST:
    CFG.update(patients=[559, 563], seeds=[41], budgets_days=[1, 'full'],
               hidden_size=8, layers=1, source_epochs=1, target_epochs=1,
               bootstrap_replicates=199)
print(json.dumps(CFG, indent=2))
print(f"Plan: {len(CFG['patients'])*len(CFG['seeds'])} source pretraining fits, "
      f"{2*len(CFG['patients'])*len(CFG['seeds'])*len(CFG['budgets_days'])} target fits.")

## 4. Stage data from Drive

Only the patients in this configuration are staged. Copies are checked against Drive by SHA-256 so stale local files are not silently reused. Data remain in the private runtime/Drive paths. Ensure your existing use of Drive complies with your data agreement.


In [ ]:
DATA_ROOT = LOCAL_ROOT / 'data'
for release, patients in cold.COHORT.items():
    for pid in patients:
        if pid not in CFG['patients']:
            continue
        for mode in ['train', 'test']:
            name = f'{pid}-ws-{mode}ing.xml'
            src = DRIVE_DATA / release / mode / name
            dst = DATA_ROOT / 'raw' / 'ohiot1dm' / release / mode / name
            if not src.is_file():
                raise FileNotFoundError(f'Missing {src}; use the same Drive dataset as the main notebook.')
            dst.parent.mkdir(parents=True, exist_ok=True)
            if not dst.is_file() or cold.digest(src) != cold.digest(dst):
                shutil.copy2(src, dst)
            assert cold.digest(src) == cold.digest(dst)
print('All required XML files staged and verified.')


## 5. Preflight, train, evaluate and resume

Before any GPU training, **every patient/budget is checked** for enough contiguous fitting/validation/test windows and a fixed test set. If a budget is infeasible, the run stops with `preflight.json`; do not silently drop that patient. Inspect coverage before deciding on a revised protocol, and document the change.

Training prints progress every epoch. A completed source checkpoint is cached once per target/seed. Each paired RL/TL budget job is copied to Drive, with its completion marker written last. A disconnect can require repeating the currently unfinished source or paired target job, but completed work is reused.

After reconnecting, run the notebook again with identical settings. Different data, code, configurations or recorded package versions use a separate fingerprinted folder. Avoid simultaneous sessions writing the same run.


In [ ]:
import time
started = time.perf_counter()
LOCAL_RUN, DRIVE_RUN = cold.run(
    DATA_ROOT, SUPPORT_DIR, CFG,
    local_root=LOCAL_ROOT / 'runs',
    drive_root=DRIVE_RESULTS / 'cold_start_followup',
    device=DEVICE)
print(f'Finished in {(time.perf_counter()-started)/3600:.2f} hours.')
print('Results:', DRIVE_RUN)


## 6. Review all budgets

- `summary/budget_summary.csv`: equal-patient MAE/RMSE, transfer benefit, joint patient-and-seed intervals, and the additional benefit relative to this experiment's full-history control.
- `summary/patient_seed_metrics.csv`: each patient's results, sequence counts and persistence reference.
- `summary/history_budget_curve.png`: all budgets with pointwise confidence intervals.
- `preflight.json`: actual history intervals, usable windows, normalization statistics and test dates.
- `jobs/`: predictions and validation histories for both regimes.
- `pretraining/`: source-only checkpoints, donor IDs and histories.
- `protocol.json`: exact configuration and data/code fingerprints.

Confidence intervals resample patients and shared seeds, using identical draws for all budgets. Wilcoxon tests use 12 patient means, with BH across both metrics and all budgets in this run; budgets are not independent replications. If adding architectures/horizons, combine the testing family before making joint significance claims.

**Interpretation:** stronger benefit with short history would support a cold-start contribution. It does not establish that every new patient benefits, that full-history gains were underestimated, or that MAE and RMSE percentages are interchangeable. Inspect absolute errors, persistence and seed variability. Report null/negative results as well as improvements.


In [ ]:
import pandas as pd
from IPython.display import display, Markdown, Image
display(Markdown((LOCAL_RUN / 'summary' / 'REPORT.md').read_text()))
display(pd.read_csv(LOCAL_RUN / 'summary' / 'budget_summary.csv'))
display(Image(filename=str(LOCAL_RUN / 'summary' / 'history_budget_curve.png')))
metrics = pd.read_csv(LOCAL_RUN / 'summary' / 'patient_seed_metrics.csv')
display(metrics.groupby('budget_days')[['n_train', 'n_validation', 'n_test']].agg(['min','median','max']))
print('Download article candidates from:', DRIVE_RUN / 'summary')


## 7. Transfer versus regular learning across history budgets

`history_budget_curve.png` above plots the relative benefit. This adds the
comparison the cold-start question is actually asking for — both regimes, at
every budget, on one axis:

- **(a)** absolute error for RL and TL with the persistence anchor, so a large
  relative gain on a model that is itself worse than a no-change forecast is
  visible as exactly that;
- **(b)** the paired RL − TL difference with its interval and a zero line;
- **(c)** the same benefit per patient, since a cohort mean carried by two
  patients is not a cohort result.

Intervals come from the same joint patient-and-seed resampling used in
`summary/budget_summary.csv`, with the same draws applied to both regimes. The
CSV written beside the figure adds `additional_vs_control`: each budget's
benefit minus the full-history benefit under identical draws, which is the
paired form of "does transfer help more when there is less data".

`RUN/experiments/plot_cold_start_tl_vs_rl.py` takes any finished run directory,
so it also draws runs produced in an earlier session, and from a shell:

```
python RUN/experiments/plot_cold_start_tl_vs_rl.py --run-dir <run> [--metric rmse] [--pdf]
```


In [ ]:
from IPython.display import display, Image

METRIC  = 'mae'        # 'rmse' for the same figure on squared-error terms
RUN_DIR = LOCAL_RUN    # any finished run directory, including one restored from Drive

result = subprocess.run(
    [sys.executable, str(PLOT_SCRIPT), '--run-dir', str(RUN_DIR), '--metric', METRIC,
     '--bootstrap-count', str(CFG['bootstrap_replicates']),
     '--resampling-seed', '42', '--pdf'],
    capture_output=True, text=True)
print(result.stdout or result.stderr)
if result.returncode != 0:
    raise SystemExit(f'Figure failed:\n{result.stderr}')

# Mirror to Drive beside the rest of the summary, so the figure survives the runtime.
for name in (f'cold_start_tl_vs_rl_{METRIC}.png',
             f'cold_start_tl_vs_rl_{METRIC}.pdf',
             f'cold_start_tl_vs_rl_{METRIC}.csv'):
    shutil.copy2(RUN_DIR / 'summary' / name, DRIVE_RUN / 'summary' / name)
print('Copied to', DRIVE_RUN / 'summary')

display(Image(filename=str(RUN_DIR / 'summary' / f'cold_start_tl_vs_rl_{METRIC}.png')))